In [0]:
# Imports

from pyspark.sql.window import Window
from pyspark.sql import functions as F

In [0]:
# Preparação dos endereços das tabelas

catalog = "atividade_2"

bronze = "bronze"
silver = "silver"
gold = "gold"

bronze_schema = f"{catalog}.{bronze}"
silver_schema = f"{catalog}.{silver}"
gold_schema = f"{catalog}.{gold}"

landing_path = f"/Volumes/{catalog}/{bronze}/Landing"

print(catalog)
print(bronze_schema)
print(silver_schema)
print(gold_schema)
print(landing_path)

atividade_2
atividade_2.bronze
atividade_2.silver
atividade_2.gold
/Volumes/atividade_2/bronze/Landing


## Dim Movies

In [0]:
df_info_silver = spark.table(f"{silver_schema}.tb_info_filmes")

df_dim_movies = (
    df_info_silver
        .select(
            F.monotonically_increasing_id().cast('BIGINT').alias("sk_movie_id"),
            F.col("id_filme").cast("string"),
            "titulo",
            "data_lancamento",
            "ano_lancamento",
            "duracao_minutos",
            "idioma_original",
            "status_filme",
            "sinopse"
        )
)

df_dim_movies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_movies")

print(f"dim_movies: {df_dim_movies.count()} linhas")

display(df_dim_movies.limit(5))

dim_movies: 97879 linhas


sk_movie_id,id_filme,titulo,data_lancamento,ano_lancamento,duracao_minutos,idioma_original,status_filme,sinopse
0,14564,Rings,2017-02-01,2017,102,en,Lançado,"\Julia becomes worried about her boyfriend Holt when he explores the dark urban legend of a mysterious videotape said to kill the watcher seven days after viewing. She sacrifices herself to save her boyfriend and in doing so makes a horrifying discovery: there is a """"\""""movie within the movie""""\"""" that no one has ever seen before.""""""""First you watch it. Then you die."
1,32471,Mixtape,2021-12-03,2021,94,en,Lançado,null
2,38258,Grizzly II: Revenge,2020-02-17,2020,74,en,Lançado,\All hell breaks loose when a giant grizzly
3,38492,Billy Joel - Live at Yankee Stadium,2022-06-22,2022,86,en,Lançado,Billy Joel plays his greatest hits in the Big Apple.
4,38700,Bad Boys for Life,2020-01-15,2020,124,en,Lançado,"Marcus and Mike are forced to confront new threats, career changes, and midlife crises as they join the newly created elite team AMMO of the Miami police department to take down the ruthless Armando Armas, the vicious leader of a Miami drug cartel."


## Fact Movies Performance

In [0]:
df_engajamento_silver = spark.table(f"{silver_schema}.tb_metricas_engajamento")
df_financeiro_silver = spark.table(f"{silver_schema}.tb_financeiro_filmes")

df_dim_movies_lk = spark.table(f"{gold_schema}.dim_movies").select("sk_movie_id", "id_filme")

df_fact_movies_performance = (
    df_engajamento_silver
        .join(df_dim_movies_lk, on="id_filme", how="inner")
        .join(df_financeiro_silver, on="id_filme", how="inner")
        .select(
            "sk_movie_id",
            "orcamento_usd",
            "receita_usd",
            "lucro_usd",
            "orcamento_brl",
            "receita_brl",
            "lucro_brl",
            "popularidade",
            "nota_media_tmdb",
            "qtd_votos_tmdb",
            "nota_media_imdb",
            "qtd_votos_imdb"
        )
)

df_fact_movies_performance.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.fact_movies_performance")

print(f"fact_movies_performance: {df_fact_movies_performance.count()} linhas")

display(df_fact_movies_performance.limit(5))

fact_movies_performance: 97742 linhas


sk_movie_id,orcamento_usd,receita_usd,lucro_usd,orcamento_brl,receita_brl,lucro_brl,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
31636,58000000.00,null,null,289623000.00,null,null,72.735,7.606,28894,8.0,1270339
63237,150000000.00,490719763.00,340719763.00,749025000.00,2450409136.54,1701384136.54,41.094,7.16,8030,7.4,329939
31604,175000000.00,966550600.00,791550600.00,873862500.00,4826470421.10,3952607921.10,33.274,6.86,7668,7.3,307846
5777,15000000.00,93411426.00,78411426.00,74902500.00,466449955.73,391547455.73,16.735,7.521,7037,7.5,317303
63390,200000000.00,66800000.00,-133200000.00,998700000.00,333565800.00,-665134200.00,43.222,6.906,6280,5.8,168602


## Dim Genres e Bridge Movie Genre

In [0]:
df_generos_silver = spark.table(f"{silver_schema}.tb_generos")

df_dim_generos = (
    df_generos_silver
        .dropDuplicates(["genero"])
        .select(
            F.monotonically_increasing_id().cast('BIGINT').alias("sk_genre_id"), 
            F.col("genero").alias("nome_genero")
        )
)

df_bridge_movie_genre = (
    df_generos_silver
        .select(
            "id_filme", 
            F.col("genero").alias("nome_genero")
        )
        .withColumn("id_filme", F.col("id_filme").cast("string"))
        .join(df_dim_movies_lk, on="id_filme", how="inner")
        .join(df_dim_generos, on="nome_genero", how="inner")
        .select("sk_genre_id", "sk_movie_id")
)

In [0]:
df_dim_generos.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_generos")

print(f"dim_generos: {df_dim_generos.count()} linhas")

display(df_dim_generos.limit(5))

dim_generos: 19 linhas


sk_genre_id,nome_genero
0,Action
1,Adventure
2,Animation
3,Comedy
4,Crime


In [0]:
df_bridge_movie_genre.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_genre")

print(f"bridge_movie_genre: {df_bridge_movie_genre.count()} linhas")

display(df_bridge_movie_genre.limit(5))

bridge_movie_genre: 142229 linhas


sk_genre_id,sk_movie_id
10,31942
10,12595
10,31629
10,56331
10,63474


## Dim People, Dim Companies e Bridges com Movie

In [0]:
df_pessoas_empresas_silver = spark.table(f"{silver_schema}.tb_pessoas_empresas")

df_dim_people = (
    df_pessoas_empresas_silver
        .where(F.col("tipo_entidade").isin(["Ator", "Diretor", "Roteirista"]))
        .dropDuplicates(["entidade", "tipo_entidade"])
        .select(
            F.monotonically_increasing_id().cast("BIGINT").alias("sk_person_id"),
            F.col("entidade").alias("nome_pessoa"),
            F.col("tipo_entidade").alias("tipo_pessoa")
        )
)

df_dim_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_people")

print(f"dim_people: {df_dim_people.count()} linhas")

display(df_dim_people.limit(5))

dim_people: 421627 linhas


sk_person_id,nome_pessoa,tipo_pessoa
0,SAM HAYGARTH,Ator
1,BRITNE OLDFORD,Ator
2,MICHELLE RODRIGUEZ,Ator
3,KIMBERLY ELISE,Ator
4,SOPHIE TURNER,Ator


In [0]:
df_dim_companies = (
    df_pessoas_empresas_silver
        .where(F.col("tipo_entidade") == "Produtora")
        .dropDuplicates(["entidade"])
        .select(
            F.monotonically_increasing_id().cast("BIGINT").alias("sk_company_id"),
            F.col("entidade").alias("nome_produtora"),
        )
)

df_dim_companies.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_companies")

print(f"dim_companies: {df_dim_companies.count()} linhas")

display(df_dim_companies.limit(5))

dim_companies: 46224 linhas


sk_company_id,nome_produtora
0,DUNE ENTERTAINMENT
1,ATLAS ENTERTAINMENT
2,PERFECT WORLD PICTURES
3,UFA
4,STAGE 6 FILMS


In [0]:
df_dim_people_lk = spark.table(f"{gold_schema}.dim_people")

df_bridge_movie_person = (
    df_pessoas_empresas_silver
        .join(df_dim_movies_lk, on="id_filme", how="inner")
        .withColumns({
            "nome_pessoa": F.col("entidade"),
            "tipo_pessoa": F.col("tipo_entidade")
        })
        .join(df_dim_people_lk, on=["nome_pessoa", "tipo_pessoa"], how="inner")
        .select(
            F.col("sk_movie_id"),
            F.col("sk_person_id")
        )
)

df_bridge_movie_person.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_person")

print(f"bridge_movie_person: {df_bridge_movie_person.count()} linhas")

display(df_bridge_movie_person.limit(5))

bridge_movie_person: 787304 linhas


sk_movie_id,sk_person_id
70543,52657
9089,368856
32381,158097
65382,184754
31706,316194


In [0]:
df_dim_companies_lk = spark.table(f"{gold_schema}.dim_companies")

df_bridge_movie_company = (
    df_pessoas_empresas_silver
        .join(df_dim_movies_lk, on="id_filme", how="inner")
        .withColumns({
            "nome_produtora": F.col("entidade"),
            "tipo_pessoa": F.col("tipo_entidade")
        })
        .join(df_dim_companies_lk, on=["nome_produtora"], how="inner")
        .select(
            F.col("sk_movie_id"),
            F.col("sk_company_id")
        )
)

df_bridge_movie_company.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.bridge_movie_company")

print(f"bridge_movie_company: {df_bridge_movie_company.count()} linhas")

display(df_bridge_movie_company.limit(5))

bridge_movie_company: 139280 linhas


sk_movie_id,sk_company_id
18982,32228
52388,14487
54337,10967
72946,10967
7414,10967


## Dim reviews

In [0]:
df_avaliacoes_usuarios_silver = spark.table(f"{silver_schema}.tb_avaliacoes_usuarios")

df_dim_reviews = (
    df_avaliacoes_usuarios_silver
        .groupBy("id_filme")
        .agg(
            F.count("id_filme").alias("qtd_avaliacoes_usuarios"),
            F.round(F.avg(F.col("nota_usuario")), 2).alias("nota_media_usuarios")
        ).withColumn("sk_review_id", F.monotonically_increasing_id().cast('BIGINT'))
        .join(df_dim_movies_lk, how="left", on="id_filme")
        .select(
            F.col("id_filme"),
            F.col("sk_review_id"),
            F.col("sk_movie_id"),
            F.col("qtd_avaliacoes_usuarios"),
            F.col("nota_media_usuarios")
        )
)

df_dim_reviews.write.format("delta").mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable(f"{gold_schema}.dim_reviews")

print(f"dim_reviews: {df_dim_reviews.count()} linhas")

display(df_dim_reviews.limit(5))

dim_reviews: 27667 linhas


id_filme,sk_review_id,sk_movie_id,qtd_avaliacoes_usuarios,nota_media_usuarios
615968,0,44235,9,4.8
822511,1,52264,9,4.9
715959,2,16653,9,null
457591,3,35637,18,7.65
628479,4,76472,9,3.7
